# Day 3 — The Pattern Finder 🔍

> **Mission briefing:** A crate just landed in the Lab with no label — thousands of measurements and *nobody* to tell us what they mean. No answer key, no teacher. Your job: find the hidden structure in the data on your own. This is **unsupervised learning**, and by tonight it'll power your Studio's newest module.

**Previously in the Lab:** yesterday you trained the 🐧 **Prediction Machine** — a classifier that learned penguin species *from labeled examples*. Today, the labels are gone.

**Today you will:**
- See the difference between **supervised** (with an answer key) and **unsupervised** (without) learning
- Run **k-means** to discover groups no one told the machine about
- Watch the machine **rediscover the penguin species** with zero labels
- Use the **elbow method** to guess how many groups exist
- Build a **photo color-compressor** with k-means
- Squash 4-D and 64-D data into 2-D pictures with **PCA** and **t-SNE**

**Today you unlock:** 🔓 **Pattern Finder** — upload any photo and it finds the dominant colors, repainting the image with just a handful of them.

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║ SETUP — run me first! (identical in every Lab notebook)           ║
# ╚═══════════════════════════════════════════════════════════════════╝
import os, sys, pathlib

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("SMOKE_TEST", "0") == "1"   # tiny fast run for automated tests

REPO_URL = "https://github.com/CHANGE-ME/ai-studio-internship"  # instructor: set once

if IN_COLAB:
    if not pathlib.Path("/content/ai-studio-internship").exists():
        !git clone -q {REPO_URL} /content/ai-studio-internship
    %cd /content/ai-studio-internship/notebooks/day03

HERE = pathlib.Path.cwd()
REPO = HERE.parents[1]                       # .../notebooks/dayNN -> repo root
DATA_DIR = pathlib.Path(os.environ.get("COURSE_DATA_DIR", REPO / "data"))
SAMPLES_DIR = REPO / "data_samples"          # small datasets shipped with the repo
MODELS_DIR = REPO / "ai_studio" / "models"   # where your Studio modules unlock
for p in (REPO / "data", MODELS_DIR):
    p.mkdir(parents=True, exist_ok=True)

import random
import numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)
print(f"✅ Setup done | Colab: {IN_COLAB} | Smoke test: {SMOKE}")
print(f"   data: {DATA_DIR}")

## 1 · Two ways to learn

Yesterday every penguin came with its species written on the tag. The machine's job was easy to *grade*: guess the species, peek at the tag, correct itself. That's **supervised learning** — learning **with an answer key**.

Today the tags are gone. We have measurements, but nothing to check our guesses against. So we change the question. Instead of *"what species is this?"* we ask:

> **"Which of these penguins resemble each other?"**

Finding groups and structure **without any labels** is **unsupervised learning**. There's no single "right answer" to copy — the machine has to notice patterns by itself.

It's not a toy. Unsupervised learning is how real systems:
- **group customers** into look-alike audiences no one defined in advance,
- **flag anomalies** — a fraudulent transaction is the point that fits no group,
- **compress images** by replacing millions of colors with a handful (you'll do exactly this today).

Let's open the crate.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# The crate: penguin measurements, with the species column still attached so
# we can peek LATER — but the machine will never get to see it.
penguins = pd.read_csv(SAMPLES_DIR / "penguins.csv").dropna().reset_index(drop=True)
print(f"{len(penguins)} penguins in the crate")
penguins.head()

## 2 · How k-means thinks — the pizza-shop story

Imagine three pizza shops opening in a new city. Nobody has a map of where customers live, so the shops play a simple game, over and over:

1. **Assign:** every customer orders from the *nearest* shop.
2. **Update:** each shop moves to the *center* of the customers it just served.

Do that a few times and the shops drift until they sit right in the middle of three natural neighborhoods. Customers stop switching. Done.

That's **k-means**. Swap "pizza shop" for **cluster center** (a *centroid*) and "customer" for **data point** and you have the whole algorithm. The `k` is how many shops (clusters) you decide to place.

Let's watch it happen on two penguin measurements — **bill length** and **flipper length** — standardized so neither one dominates just because it's measured in bigger numbers.

In [ ]:
# Two measurements, standardized (mean 0, spread 1) so both count equally.
feat2 = penguins[["bill_length_mm", "flipper_length_mm"]].to_numpy(dtype=float)
X2 = (feat2 - feat2.mean(axis=0)) / feat2.std(axis=0)

# Start the 3 "shops" (centroids) at 3 random penguins.
rng = np.random.default_rng(SEED)
centroids = X2[rng.choice(len(X2), size=3, replace=False)].copy()

fig, axes = plt.subplots(2, 3, figsize=(11, 6.5))
for step, ax in enumerate(axes.flat):
    # 1) ASSIGN: each point joins its nearest centroid
    dists = np.linalg.norm(X2[:, None, :] - centroids[None, :, :], axis=2)  # (N, 3)
    labels = dists.argmin(axis=1)

    # draw the current picture
    ax.scatter(X2[:, 0], X2[:, 1], c=labels, cmap="viridis", s=12)
    ax.scatter(centroids[:, 0], centroids[:, 1], c="red", marker="X",
               s=220, edgecolor="black", linewidth=1.5)
    ax.set_title(f"iteration {step}")
    ax.set_xlabel("bill length (standardized)")
    ax.set_ylabel("flipper length (standardized)")

    # 2) UPDATE: each centroid hops to the mean of its assigned points
    for j in range(3):
        if np.any(labels == j):
            centroids[j] = X2[labels == j].mean(axis=0)

fig.suptitle("k-means converging: the red X's drift to the center of their groups")
fig.tight_layout()
plt.show()

### 🧪 Exercise 1 — Let scikit-learn do the clustering

You watched the loop by hand; now use the real thing. `KMeans` runs those assign/update steps for you until the centroids stop moving.

- Create `KMeans(n_clusters=3, n_init=10, random_state=SEED)` and fit it to `X2`.
- Store each point's cluster number in `clusters`.

`n_init=10` just means "try 10 random starts and keep the best" — k-means can land in a bad spot if it starts unlucky. You should see three clean blobs.

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=3, n_init=10, random_state=SEED)
# ==================== YOUR CODE HERE ====================
### HINT: km.fit_predict(X2) fits the model AND returns a cluster number per point
...

plt.figure(figsize=(6, 5))
plt.scatter(X2[:, 0], X2[:, 1], c=clusters, cmap="viridis", s=15)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            c="red", marker="X", s=220, edgecolor="black", linewidth=1.5)
plt.xlabel("bill length (standardized)")
plt.ylabel("flipper length (standardized)")
plt.title("k-means found 3 groups — with no labels")
plt.show()

### 🧪 Exercise 2 — The reveal: did it find the species?

Here's the moment. The machine never saw a single species label — it only grouped penguins by shape. So did its groups mean anything?

- Build a table comparing the machine's `clusters` against the **true species** with `pd.crosstab`.
- Read it: does each cluster line up mostly with one species?

Use `pd.crosstab(clusters, species_true, ...)`.

In [ ]:
species_true = penguins["species"].to_numpy()

# ==================== YOUR CODE HERE ====================
### HINT: pd.crosstab(clusters, species_true) counts how often each pairing occurs
...
print(comparison)

**Sit with that for a second.** Nobody told the machine what a "Gentoo" is. It just noticed that some penguins clump together by bill and flipper size — and those clumps *are* the species. The structure was hiding in the numbers all along; k-means only had to look.

(The match isn't perfect — a few penguins sit on the border between groups, and two of the species really do overlap in just these two measurements. Later we'll hand k-means *all four* measurements and the picture sharpens.)

## 3 · How many groups are there, really?

We *told* k-means to find 3. But what if we didn't know? Real mystery crates don't come with the number of groups printed on the side.

Here's a trick. **Inertia** measures how tightly points hug their centroid — smaller is tighter. Add more clusters and inertia always drops (with one cluster per point it hits zero). But watch *how fast* it drops: there's usually a `k` where adding another cluster suddenly stops helping much. The curve bends like an elbow, and that bend is nature's hint at the true number of groups.

### 🧪 Exercise 3 — Find the elbow

- For each `k` from 1 to 8, fit `KMeans` and record its `.inertia_`.
- Plot inertia versus `k` and look for the bend.

You should see the curve drop steeply, then flatten around **k = 3** — exactly the number of penguin species.

In [ ]:
ks = range(1, 9)
inertias = []
# ==================== YOUR CODE HERE ====================
### HINT: after fitting, km_k.inertia_ is the tightness number to append
...

plt.figure(figsize=(6, 4))
plt.plot(list(ks), inertias, "o-")
plt.xlabel("k (number of clusters)")
plt.ylabel("inertia (lower = tighter clusters)")
plt.title("The elbow method — where does the curve bend?")
plt.show()

## 4 · The visual wow: a photo is just a pile of colors

Time for the module you're unlocking. Here's a fresh way to look at an image: **a color is just a point in 3-D space** — its red, green, and blue amounts. A photo with a million pixels is a cloud of a million points floating in that RGB cube.

So what happens if we run k-means on the pixels of a photo? It finds the `k` most representative colors — the photo's **palette** — and we repaint every pixel with its nearest palette color. Fewer colors, almost the same picture. That's **color quantization**, and it's a genuine form of image compression.

Let's grab a sample photo that ships with scikit-learn (no download needed).

In [ ]:
from sklearn.datasets import load_sample_image

china = load_sample_image("china.jpg") / 255.0     # scale RGB from 0..255 to 0..1
print("image shape:", china.shape, " (height, width, RGB)")

pixels = china.reshape(-1, 3)                        # one row per pixel
print("that's", len(pixels), "pixels, each a point in 3-D color space")


def palette_bar(centers, labels, width=400, height=60):
    """Draw the palette as a horizontal bar, widest (most common) color first."""
    counts = np.bincount(labels, minlength=len(centers))
    order = np.argsort(-counts)
    bar = np.zeros((height, width, 3))
    x = 0
    for idx in order:
        w = int(round(counts[idx] / counts.sum() * width))
        bar[:, x:x + w] = centers[idx]
        x += w
    bar[:, x:] = centers[order[-1]]                  # fill any rounding gap
    return bar


plt.figure(figsize=(6, 4))
plt.imshow(china)
plt.title("the original photo")
plt.axis("off")
plt.show()

### 🧪 Exercise 4 — Compress the photo to 8 colors

Fitting k-means on all ~270,000 pixels is slow, so we use a classic shortcut: **fit on a random sample, then predict every pixel.** The palette barely changes, but it runs in a snap.

- Fit `KMeans(n_clusters=8, n_init=4, random_state=SEED)` on the 20,000-pixel `sample`.
- Use the fitted model to `.predict` a cluster for **every** pixel in `pixels` (store it in `labels_all`).

Then run the given code to repaint the photo. Eight colors — see how close it still looks.

In [ ]:
rng = np.random.default_rng(SEED)
sample = pixels[rng.choice(len(pixels), size=20_000, replace=False)]

km_img = KMeans(n_clusters=8, n_init=4, random_state=SEED)
# ==================== YOUR CODE HERE ====================
### HINT: fit on `sample` (fast), then km_img.predict(pixels) to label ALL pixels
...

palette = km_img.cluster_centers_
repainted = palette[labels_all].reshape(china.shape)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].imshow(china);     axes[0].set_title("original (thousands of colors)")
axes[1].imshow(repainted); axes[1].set_title("repainted with only 8 colors")
for ax in axes:
    ax.axis("off")
plt.show()

plt.figure(figsize=(6, 1.2))
plt.imshow(palette_bar(palette, labels_all))
plt.title("the 8-color palette k-means chose")
plt.axis("off")
plt.show()

### 🧪 Exercise 5 — How few colors can you get away with?

Let's make `k` a dial. The `quantize` helper below repaints the photo with any number of colors. You fill in the one line that displays each version.

- Inside the loop, call `quantize(k)` and show it with `ax.imshow(...)`.
- Compare k = 3, 8, and 16.

Then answer in the text cell below: what do you *lose* at k = 3? What barely changes between k = 8 and k = 16?

In [ ]:
def quantize(k, n_sample=20_000):
    """Repaint `china` using only k colors. Returns an (H, W, 3) image."""
    samp = pixels[rng.choice(len(pixels), size=n_sample, replace=False)]
    model = KMeans(n_clusters=k, n_init=4, random_state=SEED).fit(samp)
    return model.cluster_centers_[model.predict(pixels)].reshape(china.shape)


fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, k in zip(axes, [3, 8, 16]):
    # ==================== YOUR CODE HERE ====================
    ### HINT: quantize(k) returns the repainted image; hand it to ax.imshow(...)
    ...
    ax.set_title(f"k = {k} colors")
    ax.axis("off")
plt.show()

**Your turn to describe** *(double-click to edit):*

- At **k = 3**, the photo loses ...
- Between **k = 8** and **k = 16**, ...

💡 You just built a photo compressor — and the engine behind your Studio's 🎨 **Pattern Finder** module. Same k-means, running live on whatever photo you upload.

## 5 · Seeing in 4-D: PCA casts a shadow

Our penguins actually have **four** measurements, not two. But a screen is flat — how do you plot a 4-D animal?

Think of shadows. A 3-D object casts a 2-D shadow on the wall, and if you rotate it to *just the right angle*, the shadow keeps almost everything that makes the shape recognizable. **PCA** (Principal Component Analysis) finds that best angle automatically: it squashes many dimensions down to a few while throwing away as little information as possible.

### 🧪 Exercise 6 — Flatten 4-D penguins onto a 2-D page

- Standardize all **four** measurements, then fit `PCA(n_components=2)` and transform them into `coords`.
- We'll color the picture by the *true* species — but only to check the result. **PCA never sees the labels;** it arranges the penguins from the measurements alone.
- Print `explained_variance_ratio_` and its sum: the fraction of the original information the 2-D shadow kept.

You should keep well over 80% of the information in just two numbers per penguin.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

feat4 = penguins[["bill_length_mm", "bill_depth_mm",
                  "flipper_length_mm", "body_mass_g"]].to_numpy(dtype=float)
X4 = StandardScaler().fit_transform(feat4)

# ==================== YOUR CODE HERE ====================
### HINT: pca.fit_transform(X4) returns the 2-D coordinates
...

species_codes, species_labels = pd.factorize(penguins["species"])
plt.figure(figsize=(7, 5))
for c, name in enumerate(species_labels):
    m = species_codes == c
    plt.scatter(coords[m, 0], coords[m, 1], s=15, label=name)
plt.xlabel("principal component 1")
plt.ylabel("principal component 2")
plt.title("4 measurements squashed into 2 — colored by true species")
plt.legend()
plt.show()

print("information kept per axis:", np.round(pca.explained_variance_ratio_, 3))
print("total information kept:   ", round(float(pca.explained_variance_ratio_.sum()), 3))

## 6 · A map of handwriting with t-SNE

PCA is fast and honest, but it can only *rotate and flatten* — straight-line moves. Some data is tangled in ways that need a curvier map. **t-SNE** builds one: it pulls similar points close together and shoves different ones apart, untangling structure PCA can miss. It's the go-to tool for *seeing* high-dimensional data.

Let's aim it at something with 64 dimensions: tiny 8×8 images of handwritten digits (built right into scikit-learn — no download).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print(digits.images.shape, "→ 1797 images, each 8×8 pixels (64 numbers)")

fig, axes = plt.subplots(2, 8, figsize=(10, 2.8))
for ax, img, label in zip(axes.flat, digits.images, digits.target):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(str(label))
    ax.axis("off")
fig.suptitle("a handful of the digits")
plt.show()

In [ ]:
from sklearn.manifold import TSNE

# t-SNE is slower than PCA. Under a smoke test we use only the first 400 images.
n_show = 400 if SMOKE else len(digits.data)
X_digits = digits.data[:n_show]
y_digits = digits.target[:n_show]

print(f"Running t-SNE on {n_show} images… this takes ~20-40s on CPU. Hang tight ☕")
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30)
embedding = tsne.fit_transform(X_digits)
print("Done — here comes the map.")

### 🧪 Exercise 7 — Draw the map

`embedding` holds a 2-D point for each digit image. Color each point by which digit it actually is and see whether t-SNE sorted them.

- Scatter `embedding[:, 0]` vs `embedding[:, 1]`, colored by `y_digits` (use `cmap="tab10"`).
- Keep the returned scatter object so we can attach a colorbar.

You should see roughly ten islands — one per digit — that nobody told t-SNE about.

In [ ]:
plt.figure(figsize=(8, 6))
# ==================== YOUR CODE HERE ====================
### HINT: pass c=y_digits and cmap="tab10" to color each point by its digit
...
plt.colorbar(scatter, label="digit")
plt.xlabel("t-SNE axis 1")
plt.ylabel("t-SNE axis 2")
plt.title("1797 handwritten digits, sorted into islands with zero labels")
plt.show()

**1797 images, 64 pixels each — and the map sorts them into islands, one per digit, without ever being told what a digit is.** The pixels alone carry the structure; t-SNE just makes it visible.

Soon you'll go one step further and **train a network to actually read these digits.**

## 7 · Optional: UMAP, t-SNE's faster cousin

**UMAP** does a similar untangling job, often faster and better at keeping the big-picture layout. It's an extra library, so this cell skips itself politely if UMAP isn't installed — no worries either way.

In [ ]:
try:
    import umap  # optional — present in the Lab's Docker image, maybe not on Colab
    reducer = umap.UMAP(n_components=2, random_state=SEED)
    umap_emb = reducer.fit_transform(X_digits)

    plt.figure(figsize=(8, 6))
    sc = plt.scatter(umap_emb[:, 0], umap_emb[:, 1], c=y_digits, cmap="tab10", s=12)
    plt.colorbar(sc, label="digit")
    plt.xlabel("UMAP axis 1")
    plt.ylabel("UMAP axis 2")
    plt.title("the same digits, mapped by UMAP")
    plt.show()
except ImportError:
    print("UMAP isn't installed here — that's totally fine, skip this one. 🙂")

## 🔓 Unlock your Studio module

You've earned it. Save your settings and the 🎨 **Pattern Finder** switches on in your AI Studio. Put your name on your work and pick the number of colors it should reach for by default.

In [ ]:
import json

# ✏️ Make it yours:
MADE_BY = "your name here"     # ← type your name between the quotes
DEFAULT_K = 8                  # ← default number of color clusters (2–12)

config_path = MODELS_DIR / "pattern_finder.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump({"default_k": DEFAULT_K, "made_by": MADE_BY}, f, indent=2)

# make sure the Studio will find it
assert config_path.exists(), "pattern_finder.json did not save!"
print("🔓 UNLOCKED: Pattern Finder!")
print(f"   saved {config_path.name}  (default_k={DEFAULT_K}, made_by={MADE_BY})")
print("   Run your Studio to try it live on any photo:")
print("   python ai_studio/app.py   (from the repo root)")

## 🏁 Checkpoint

Look what you can do now — with **no answer key at all**:

- **k-means** to discover groups (and you watched it *rediscover the penguin species*)
- the **elbow method** to guess how many groups exist
- **color quantization** to compress a photo — now live in your 🎨 **Pattern Finder**
- **PCA** and **t-SNE** to turn 4-D and 64-D data into pictures you can actually see

**So far you've borrowed brains (Day 1) and learned the classic tricks (Days 2-3).** Tomorrow the Chief Scientist raises the stakes: **build and train a neural network from nothing but NumPy** — no libraries to hide behind. By the end, you'll never see a neural net as a black box again.

**Next: Day 4 — Inside the Brain 🧠**